# DQN — Playing Atari with Deep Reinforcement Learning (2013)

_Papers · Reinforcement Learning_

**Train a neural network to predict action values by regressing it onto a one-step Bellman target, with a replay buffer and a target network to keep the regression stable.**

---

This notebook is a practice scaffold for the **DQN — Playing Atari with Deep Reinforcement Learning (2013)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import random
from collections import deque
import torch
import torch.nn as nn
import gymnasium as gym

torch.manual_seed(0); random.seed(0)
GAMMA = 0.99

# --- 0. Sanity-check the lesson's worked TD target: gamma=0.99, r=1, done=0, Q(s')=[12,15]. ---
q_next = torch.tensor([12.0, 15.0])               # target net's next-state action values
y = 1.0 + GAMMA * (1 - 0) * q_next.max()          # 1 + 0.99 * max(12,15) = 1 + 14.85
q_pred = torch.tensor(13.0)                        # live net's Q(s,a) for the action taken
td_err = y - q_pred                                # 15.85 - 13.0 = 2.85
print("worked example:  max =", q_next.max().item(), " target y =", round(y.item(), 4),
      " TD error =", round(td_err.item(), 4), " sq loss =", round((td_err**2).item(), 4))
# worked example:  max = 15.0  target y = 15.85  TD error = 2.85  sq loss = 8.1225


# --- 1. The Q-network (Track B: nn.Linear primitives). Output = one value per action. ---
class QNet(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_act))
    def forward(self, x):
        return self.net(x)                          # shape (batch, n_act)


# --- 2. Experience-replay buffer: store transitions, return a random minibatch. ---
class Replay:
    def __init__(self, capacity=50000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = zip(*batch)
        return (torch.tensor(s, dtype=torch.float32),
                torch.tensor(a, dtype=torch.long),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(s2, dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))
    def __len__(self):
        return len(self.buf)


# --- 3. One DQN update: the TD/Bellman squared loss (Eq. 2). ---
def learn(q, q_target, opt, replay, batch=64, use_target=True):
    if len(replay) < batch:
        return
    s, a, r, s2, done = replay.sample(batch)
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)         # Q(s,a;theta) for action taken
    with torch.no_grad():
        net_for_target = q_target if use_target else q       # ABLATION (a): use live net
        q_next = net_for_target(s2).max(1).values            # max_a' Q(s',a'; theta^-)
        y = r + GAMMA * (1.0 - done) * q_next                # target; (1-done) zeros terminal bootstrap
    loss = (q_sa - y).pow(2).mean()                          # Eq. 2: squared TD error
    opt.zero_grad(); loss.backward(); opt.step()


# --- 4. Train until CartPole is solved; PRINT the return rising. ---
def train(use_target=True, use_replay=True, episodes=300, sync_every=200):
    torch.manual_seed(0); random.seed(0)
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]; n_act = env.action_space.n
    q = QNet(obs_dim, n_act); q_target = QNet(obs_dim, n_act)
    q_target.load_state_dict(q.state_dict())                 # target net starts as a copy
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)
    replay = Replay()
    eps, step, recent, history = 1.0, 0, [], []
    for ep in range(episodes):
        s, _ = env.reset(seed=ep); done = False; ep_ret = 0.0
        while not done:
            if random.random() < eps:                        # epsilon-greedy
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(q(torch.tensor(s, dtype=torch.float32)).argmax())
            s2, r, term, trunc, _ = env.step(a); done = term or trunc
            replay.push(s, a, r, s2, float(done)); s = s2; ep_ret += r; step += 1
            if use_replay:
                learn(q, q_target, opt, replay, use_target=use_target)
            else:                                            # ABLATION (b): no buffer -> train on
                one = Replay(); one.push(s, a, r, s2, float(done))  # the single latest transition
                learn(q, q_target, opt, one, batch=1, use_target=use_target)
            if use_target and step % sync_every == 0:        # periodic target-net sync
                q_target.load_state_dict(q.state_dict())
        eps = max(0.02, eps * 0.97)                           # decay exploration
        recent.append(ep_ret); avg = sum(recent[-20:]) / len(recent[-20:]); history.append(avg)
        if ep % 20 == 0:
            print(f"  ep {ep:3d}  eps {eps:.2f}  avg return (last 20): {avg:6.1f}")
        if avg >= 475:
            print(f"  -> SOLVED CartPole at episode {ep}."); break
    env.close()
    return history

print("Full DQN (replay + target net):")
full_hist = train(use_target=True, use_replay=True)
print("\nABLATION (a) -- NO target net (target from the live net, same everything else):")
notarget_hist = train(use_target=False, use_replay=True)
print("\nABLATION (b) -- NO replay (train on each transition in order):")
noreplay_hist = train(use_target=True, use_replay=False)
print("\nFull DQN     avg-return trajectory:", [round(h,1) for h in full_hist[::20]])
print("No-target    avg-return trajectory:", [round(h,1) for h in notarget_hist[::20]])
print("No-replay    avg-return trajectory:", [round(h,1) for h in noreplay_hist[::20]])
# The full DQN climbs toward ~500 and solves CartPole; both ablations rise more slowly and
# oscillate/collapse. (Exact numbers vary by hardware/seed; our small run, not the paper's
# Atari results.)

## Visualize the data & results

_Does the full DQN (experience replay + target network + the TD/Bellman loss) make the episode return rise to the solved score on CartPole, and do the two ablations (remove the target net; remove replay, same everything else) destabilize it? We train all three for the same episodes and plot the average return._

In [ ]:
# Sketch of how the three curves above are produced (full loop in the CODE cell).
# Train the full DQN and the two ablations on CartPole-v1 for the same number of
# episodes with identical net / optimizer / lr / seed, recording avg return per episode.
#
#   full_hist     = train(use_target=True,  use_replay=True)   # green: climbs to ~500 (SOLVED)
#   notarget_hist = train(use_target=False, use_replay=True)   # red:   bootstraps off live net -> oscillates
#   noreplay_hist = train(use_target=True,  use_replay=False)  # amber: correlated samples -> slow, unstable
#
# The points plotted are the per-episode average return (last 20 episodes).
# Full DQN -> smooth climb past the 475 "solved" line.
# No target net -> rises then spikes/crashes (moving target).
# No replay -> learns slowly, never settles (correlated, non-i.i.d. data).
# (Numbers are from our own run; the paper reports Atari scores -- six of seven games beaten,
#  human expert surpassed on three -- not these CartPole values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked target. With $\gamma = 0.99$, a transition has reward $r = 1.0$ and a
            non-terminal next state $s'$ ($\text{done} = 0$). The target network outputs for $s'$ the
            action values $Q(s',\text{left}) = 12.0$ and $Q(s',\text{right}) = 15.0$. The live network predicts
            $Q(s,a;\theta) = 13.0$ for the action taken. Compute the Bellman target $y$, the TD error, and the
            squared loss for this sample.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Max over next actions: $\max(12.0, 15.0) = 15.0$. — _The target uses the BEST next-state value (Eq. 1's $\max_{a'}$), computed from the target net $\theta^-$._
- Discount and add reward: $y = 1.0 + 0.99 \times 1 \times 15.0 = 15.85$. — _The $(1-\text{done}) = 1$ here keeps the bootstrap term; this is the regression label._
- TD error: $y - Q(s,a;\theta) = 15.85 - 13.0 = 2.85$. — _How wrong the live prediction was about this step &mdash; the quantity the loss squares._
- Squared loss: $2.85^2 = 8.1225$. — _Eq. 2 minimizes this; the gradient step pushes $Q(s,a;\theta)$ up toward $15.85$._

**Answer:** $y = 15.85$, TD error $= 2.85$, squared loss $= 8.1225$. The notebook recomputes
                 $1.0 + 0.99\times\max(12,15) = 15.85$ and $(15.85 - 13.0)^2 = 8.1225$.

</details>

**Problem 2.** The terminal case. Same transition as above, but now $s'$ is terminal
            ($\text{done} = 1$). What is the target $y$, and why does the next-state value drop out?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the terminal rule: $y = r + \gamma (1 - \text{done}) \max_{a'} Q(s',a';\theta^-)$ with $\text{done} = 1$. — _The $(1-\text{done})$ factor becomes $0$, zeroing the bootstrap term._
- So $y = 1.0 + 0.99 \times 0 \times 15.0 = 1.0$. — _A terminal state has no future, so the only credit is the immediate reward (Algorithm 1's terminal branch)._

**Answer:** $y = r = 1.0$. There is no future after a terminal state, so the discounted next-state value
                 is excluded &mdash; bootstrapping past the end of an episode would invent reward that does not
                 exist and inflate the values.

</details>

**Problem 3.** The ablation. Your DQN solves CartPole (return climbs toward ~500). Run two ablations,
            changing exactly one thing each time: (a) remove the target network &mdash; compute $y$ from
            the live, constantly-updating network; (b) remove experience replay &mdash; train on each
            transition the instant it happens, in order. What happens to each return curve, and what does that
            demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) Build $y$ from the live net instead of the frozen target net; keep network, replay, optimizer, and seed fixed. — _Now the regression target moves every gradient step (bootstrapping off the weights being updated) &mdash; the moving-target instability._
- (b) Replace the buffer sample with the single latest transition, used once in order; keep everything else fixed. — _Consecutive frames are highly correlated, so the net sees a stream of near-identical, non-independent examples._
- Retrain each and watch the return: both ablations rise more slowly and then oscillate or collapse, while the full DQN climbs and holds near 500. — _Each ablation removes one stabilizer, re-exposing one leg of the deadly triad; the full agent keeps both._

**Answer:** Both ablations destabilize. Removing the target network makes the regression chase a
                 target that shifts every step, so the return spikes and crashes (or diverges). Removing
                 replay trains on correlated, non-independent consecutive samples, so learning is slow and
                 unstable. Since each run changes exactly one ingredient, this isolates the two stabilizers as the
                 source of DQN's stability. The CODEVIZ panel shows this contrast.

</details>